# Semana 11 — Integração Python e SQL: Instalação, Configuração e Primeiros Passos

Nesta semana você conecta o Python direto a um banco de dados e executa comandos SQL a partir do seu próprio código — sem precisar abrir um programa de banco de dados separado toda vez.

**⚠️ Este notebook precisa rodar no VS Code local (Windows), não no Google Colab.** O motivo é simples: uma das instalações desta semana é um programa do Windows (o driver ODBC do SQLite), e o Google Colab não permite instalar programas do seu computador — ele roda numa máquina na nuvem, não na sua.

### 🧭 De onde você vem

Nas Semanas 08 a 10 você aprendeu SQL dentro de um banco de dados relacional — consultas, filtros, joins, funções e operações de CRUD (Create, Read, Update, Delete), sempre escrevendo os comandos direto numa ferramenta de banco de dados. A partir de agora, esses mesmos comandos SQL vão ser disparados **pelo seu código Python** — é a integração entre as duas linguagens que qualquer sistema real usa no dia a dia.

## Passo 0 — Preparando seu computador (faça isso só uma vez)

O seu computador ainda não tem nada instalado para o Python conversar com um banco de dados. Antes de qualquer código de verdade, você precisa instalar 2 coisas:

1. A biblioteca **`pyodbc`** — o pacote Python que sabe conversar com bancos de dados através do padrão ODBC.
2. O **driver ODBC do SQLite** — um programa do Windows que ensina o seu computador a entender arquivos `.sqlite`/`.db` através desse mesmo padrão ODBC.

Vamos ver o que acontece se você tentar usar o `pyodbc` sem ele estar instalado.

In [ ]:
import pyodbc

O erro `ModuleNotFoundError: No module named 'pyodbc'` significa exatamente o que está escrito: o Python não encontrou essa biblioteca porque ela ainda não foi instalada no seu computador. É um erro normal e esperado agora — resolve-se com uma instalação, não com uma correção de código.

Instale a biblioteca rodando a célula abaixo:

In [ ]:
%pip install pyodbc

Agora rode a importação de novo — dessa vez sem erro:

In [ ]:
import pyodbc

print("pyodbc instalado com sucesso, versão:", pyodbc.version)

### Instalando o driver ODBC do SQLite

A biblioteca `pyodbc` já está pronta, mas ela sozinha não sabe conversar com nenhum banco específico — ela precisa de um **driver** para cada tipo de banco (um driver para SQLite, outro para SQL Server, outro para MySQL, etc.). Diferente do `pyodbc`, o driver não se instala com `pip` — ele é um programa do Windows. Siga esse passo a passo:

1. Acesse **http://www.ch-werner.de/sqliteodbc/** no seu navegador.
2. Baixe o instalador de **64 bits** — o nome do arquivo é `sqliteodbc_X.X.exe` (sem "_w32" no nome; a versão de 32 bits tem "_w32" no nome do arquivo — é o erro mais comum nesta instalação, então confira antes de baixar).
3. Execute o arquivo baixado e siga o instalador clicando em "Next" até "Finish" (as opções padrão já servem).
4. Feche o VS Code completamente e abra de novo (ou pelo menos reinicie o terminal), para o Windows reconhecer o driver novo.

Depois de instalar, rode a célula abaixo para conferir se o driver apareceu:

In [ ]:
import pyodbc

print(pyodbc.drivers())

Procure **"SQLite3 ODBC Driver"** na lista impressa acima. Se ele aparecer, a instalação funcionou e você não precisa repetir esse Passo 0 de novo em nenhum outro notebook desta semana.

Se não aparecer:
1. Reinicie o computador (não só o VS Code) e rode a célula de novo.
2. Se ainda assim não aparecer, o mais provável é ter baixado a versão de 32 bits (`_w32`) por engano — volte ao passo anterior, baixe a versão de 64 bits e instale de novo.

✅ Computador configurado. Os arquivos de banco de dados que você vai usar nesta semana (`salarios.sqlite` e `chinook.db`) já estão na pasta `dataset/`, ao lado deste notebook.

## Passo 1 — Por que integrar Python e SQL?

A integração entre Python e SQL permite executar comandos SQL diretamente pelo seu código Python, dentro do banco de dados. Isso possibilita pegar o resultado de uma consulta SQL e já trazer esse resultado pronto para dentro de uma análise no Python, sem sair do seu script.

Para você acompanhar sem se preocupar em instalar um banco de dados completo, esta semana usa o **SQLite** — o mesmo banco de dados mais simples que você já viu em semanas anteriores — e a biblioteca **`pyodbc`**, que funciona com a mesma estrutura de código para vários outros bancos de dados (SQL Server, MySQL, Oracle, entre outros). Isso significa que o que você aprender aqui vale para qualquer um desses bancos, trocando só o driver — **guarde essa ideia, porque ela vai explicar uma escolha estranha lá no Passo 4.**

**Quando faz sentido usar Python + SQL, e quando só o SQL já basta?**
- Só SQL: quando você só precisa consultar ou alterar dados diretamente na ferramenta do banco de dados.
- Python + SQL: quando um sistema (uma automação, uma análise, um aplicativo) precisa ler ou escrever no banco sozinho, sem alguém abrindo uma ferramenta manualmente — inclusive para criar o próprio banco de dados por código.

## Passo 2 — Conectando ao banco de dados com pyodbc

Para o Python conversar com qualquer banco através do `pyodbc`, o caminho é sempre o mesmo, em 3 etapas: **conectar → criar um cursor → executar comandos SQL**.

O primeiro passo é montar os dados de conexão. Cada banco de dados pede um driver diferente (por isso o `pyodbc.drivers()` do Passo 0 é útil — ele mostra os drivers já instalados no seu computador):

```
dados_conexao = ("Driver={Seu_Driver};"
                "Server=SeuServidor;"
                "Database=NomeDoBanco;")
```

Repare que só o `Driver` fica entre chaves `{ }` — isso é uma exigência do próprio padrão ODBC, porque nomes de driver costumam ter espaço no meio (como `SQLite3 ODBC Driver`), e as chaves marcam onde o nome do driver começa e termina. `Server` e `Database` não precisam disso.

Se o banco pedisse login e senha, o formato ganharia mais duas partes: `"UID=SeuLogin;"` e `"PWD=SuaSenha;"`. No nosso caso, o SQLite não usa login nem senha — só o caminho do arquivo.

**E o `Server=localhost`, se o SQLite nem tem servidor?** Diferente do PostgreSQL das Semanas 08-10 (onde existia um serviço rodando e `localhost` significava algo real), o SQLite é um banco **sem servidor** — é só um arquivo. O campo `Server` continua ali porque o formato de conexão do ODBC exige ele preenchido, mas para o SQLite ele não tem efeito nenhum — quem importa de verdade é o `Database`, com o caminho do arquivo.

In [ ]:
import pyodbc

dados_conexao = ("Driver={SQLite3 ODBC Driver};"
            "Server=localhost;"
            "Database=dataset/salarios.sqlite;")

conexao = pyodbc.connect(dados_conexao)
print("Conexão bem sucedida")

Com a conexão aberta, o próximo passo é criar o que o `pyodbc` chama de **cursor** — o objeto responsável por executar os comandos SQL dentro dessa conexão.

In [ ]:
cursor = conexao.cursor()

Agora você já pode executar comandos SQL através do cursor. Existem 2 formas de trazer o resultado:

1. `cursor.execute("COMANDO_SQL")` + `cursor.fetchall()` — executa o comando e devolve os resultados como uma lista de tuplas.
2. `pd.read_sql("COMANDO_SQL", conexao)` — executa o mesmo comando e já devolve o resultado pronto como um DataFrame do Pandas.

Vamos usar a primeira forma para trazer as 10 primeiras linhas da tabela `Salaries` (a segunda forma aparece no Passo 4):

In [ ]:
cursor.execute("SELECT * FROM Salaries")
valores = cursor.fetchall()
print(valores[:10])

Depois de terminar de usar o banco, sempre feche o cursor e a conexão — isso libera o arquivo do banco de dados para outros programas (e evita erros de "arquivo em uso" numa próxima tentativa de conexão).

In [ ]:
cursor.close()
conexao.close()

## Passo 3 — Inserindo dados no banco de dados (Create)

Além de consultar (Read), o Python também pode criar registros novos dentro do banco — o C do CRUD. Desta vez a conexão aponta para outro arquivo, o `chinook.db`, um banco de dados de uma loja de mídia digital (álbuns, artistas, faixas).

In [ ]:
import pyodbc

dados_conexao = ("Driver={SQLite3 ODBC Driver};Server=localhost;Database=dataset/chinook.db")

conexao = pyodbc.connect(dados_conexao)
cursor = conexao.cursor()

O comando SQL `INSERT INTO` cria uma linha nova numa tabela. Abaixo, um álbum novo é cadastrado na tabela `albums`, associado ao artista de `ArtistId` 4 (Alanis Morissette, já cadastrada na tabela `artists`).

Depois do `INSERT`, é obrigatório salvar a alteração de fato no arquivo do banco — sem isso, ela fica só na memória. Tanto `conexao.commit()` quanto `cursor.commit()` fazem exatamente essa mesma coisa no `pyodbc` (o cursor só repassa o pedido pra conexão) — este material usa sempre `conexao.commit()`, pra não ter dúvida de qual dos dois usar.

In [ ]:
cursor.execute('''
INSERT INTO albums (Title, ArtistId)
VALUES
('Lira Rock', 4)
''')

conexao.commit()

Para confirmar que o álbum foi realmente criado, consulte de volta os últimos registros da tabela:

In [ ]:
cursor.execute("SELECT * FROM albums ORDER BY AlbumId DESC LIMIT 3")
print(cursor.fetchall())

cursor.close()
conexao.close()

### ✏️ Atividade Prática 1 — Sua vez de programar (Create)

**Contextualização:** a Chinook Records fechou uma parceria para remasterizar álbuns antigos do catálogo. O primeiro da lista é um álbum novo da banda **AC/DC** (`ArtistId` 1, já cadastrada em `artists`), que precisa entrar na tabela `albums` antes de a equipe de marketing anunciar o relançamento.

**Comando:** conecte no `chinook.db`, e insira na tabela `albums` um registro com `Title` = `'Back in Black (Remaster)'` e `ArtistId` = `1`. Depois, confirme com um `SELECT` que o álbum aparece na tabela. Use o mesmo padrão do exemplo acima (conectar → cursor → `INSERT` → `commit` → `SELECT` de confirmação → fechar).

In [ ]:
# escreva seu código aqui

import pyodbc

# Conectar ao banco de dados
dados_conexao = ("Driver={SQLite3 ODBC Driver};Server=localhost;Database=dataset/chinook.db")

conexao = pyodbc.connect(dados_conexao)

# Criar cursor
cursor = conexao.cursor()

# Insere um registro e comita
cursor.execute('''
    INSERT INTO albums (Title, ArtistId)
    VALUES (?, ?)
''', ('Back in Black (Remaster)', 1))


cursor.execute(
    "INSERT INTO albums (Title, ArtistId) VALUES (?, ?)", 
    ("Back in Black (Remaster) teste", 2)
    )

conexao.commit()

# Faz uma consulta e encerra a conexão
cursor.execute("SELECT * FROM albums ORDER BY AlbumId DESC LIMIT 3")
print(cursor.fetchall())

cursor.close()
conexao.close()

## Passo 4 — Lendo dados com Read (2 formas)

Você já leu dados no Passo 2, mas existem duas formas de fazer isso com o `pyodbc`, e vale a pena conhecer as duas:

1. **Manual**: `cursor.execute()` + `cursor.fetchall()` — você mesmo monta o DataFrame a partir do resultado.
2. **Direto com Pandas**: `pd.read_sql()` — o Pandas já devolve o resultado pronto como DataFrame, numa linha só.

Comece pela forma manual, lendo a tabela `customers` do `chinook.db`:

In [ ]:
import pyodbc

dados_conexao = ("Driver={SQLite3 ODBC Driver};Server=localhost;Database=dataset/chinook.db")

conexao = pyodbc.connect(dados_conexao)

cursor = conexao.cursor()

cursor.execute("SELECT * FROM customers")

valores = cursor.fetchall()
descricao = cursor.description
cursor.close()


Repare que o `cursor.close()` acima já aconteceu, mas `valores` e `descricao` continuam existindo — eles foram guardados em variáveis Python assim que o `execute`/`fetchall` rodou, então já não dependem mais do cursor estar aberto. Fechar o cursor só impede que você rode um *novo* comando SQL nele; não apaga o que você já salvou em variáveis.

O `cursor.description` guarda os metadados das colunas retornadas (nome, tipo, etc.) — não os dados em si. Para montar um DataFrame, você precisa dos nomes das colunas separadamente:

In [ ]:
colunas = [tupla[0] for tupla in descricao]
print(colunas)

In [ ]:
import pandas as pd

tabela_clientes = pd.DataFrame.from_records(valores, columns=colunas)
display(tabela_clientes.head())

`pd.DataFrame.from_records()` monta uma tabela a partir de uma lista de tuplas (exatamente o formato que `fetchall()` devolve) — é equivalente ao `pd.DataFrame(valores, columns=colunas)` que você já conhece, só que com um nome que deixa claro que a origem são "registros" (linhas de um banco de dados), não um dicionário.

In [ ]:
conexao.close()

Agora a forma direta, usando `pd.read_sql()` — o mesmo resultado, em muito menos código. Aqui a conexão é aberta com o módulo `sqlite3` (nativo do Python, sem precisar de driver ODBC nenhum) em vez do `pyodbc`, porque o Pandas já sabe conversar direto com um arquivo SQLite.

**Se isso funciona sem instalar nada, por que você instalou `pyodbc` + o driver no Passo 0?** Porque esse atalho com `sqlite3` só existe porque o banco É um arquivo SQLite. Se este fosse um SQL Server, MySQL ou o PostgreSQL das Semanas 08-10, esse módulo nativo não serviria — você precisaria do `pyodbc` (ou do `psycopg2`, específico do Postgres) do mesmo jeito. O Passo 0 vale para qualquer banco real que você for usar depois do curso; este atalho vale só para SQLite.

In [ ]:
import pandas as pd
import sqlite3

conexao = sqlite3.connect("dataset/chinook.db")

tabela_clientes_outra = pd.read_sql("SELECT * FROM customers", conexao)
display(tabela_clientes_outra)

conexao.close()

In [30]:
display(tabela_clientes_outra.head())

,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,NaN,Theodor-Heuss-Straße 34,Stuttgart,NaN,Germany,70174,+49 0711 2842222,NaN,leonekohler@surfeu.de,5
2,3,François,Tremblay,NaN,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,NaN,ftremblay@gmail.com,3
3,4,Bjørn,Hansen,NaN,Ullevålsveien 14,Oslo,NaN,Norway,0171,+47 22 44 22 22,NaN,bjorn.hansen@yahoo.no,4
4,5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,NaN,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


### ✏️ Atividade Prática 2 — Sua vez de programar (Read)

**Contextualização:** o time de suporte da Chinook Records precisa entrar em contato com todos os clientes brasileiros para avisar sobre uma promoção. Eles pediram uma lista com nome e e-mail desses clientes.

**Comando:** usando a forma que você preferir (manual com `cursor`, ou direto com `pd.read_sql`), consulte na tabela `customers` do `chinook.db` as colunas `FirstName`, `LastName` e `Email` de todos os clientes onde `Country = 'Brazil'`.

In [35]:
# escreva seu código aqui
import pandas as pd
import sqlite3

conexao = sqlite3.connect("dataset/chinook.db")

tabela_lista_email = pd.read_sql("SELECT FirstName, LastName, Email, Country FROM customers WHERE Country = 'Brazil'", conexao)
display(tabela_lista_email)

conexao.close()

,FirstName,LastName,Email,Country
0,Luís,Gonçalves,luisg@embraer.com.br,Brazil
1,Eduardo,Martins,eduardo@woodstock.com.br,Brazil
2,Alexandre,Rocha,alero@uol.com.br,Brazil
3,Roberto,Almeida,roberto.almeida@riotur.gov.br,Brazil
4,Fernanda,Ramos,fernadaramos4@uol.com.br,Brazil


## Passo 5 — Atualizando registros (Update)

O comando SQL `UPDATE` altera valores de linhas que já existem. Abaixo, o e-mail de um cliente cadastrado no `chinook.db` é corrigido.

In [36]:
import pyodbc

dados_conexao = ("Driver={SQLite3 ODBC Driver};Server=localhost;Database=dataset/chinook.db")

conexao = pyodbc.connect(dados_conexao)

cursor = conexao.cursor()

In [37]:
cursor.execute('''
UPDATE customers SET Email='lira@embraer.com.br' WHERE Email='luisg@embraer.com.br'
''') # executar o comando SQL

conexao.commit() # perpetuar no banco as alterações

cursor.close()
conexao.close() # finalizar a conexão

### ✏️ Atividade Prática 3 — Sua vez de programar (Update)

**Contextualização:** a cliente de `CustomerId` 2 (`leonekohler@surfeu.de`) ligou avisando que mudou de cidade e agora mora em `'Berlin'`.

**Comando:** conecte no `chinook.db` e atualize a coluna `City` da tabela `customers` para `'Berlin'`, filtrando por `CustomerId = 2`. Depois, confirme com um `SELECT` que a cidade mudou.

In [2]:
import pyodbc

dados_conexao = ("Driver={SQLite3 ODBC Driver};Server=localhost;Database=dataset/chinook.db")

conexao = pyodbc.connect(dados_conexao)

cursor = conexao.cursor()

cursor.execute('''
UPDATE customers SET City='Berlin' WHERE CustomerId = 2
''') # executar o comando SQL

conexao.commit() # perpetuar no banco as alterações

cursor.execute('''
SELECT * FROM customers WHERE CustomerId = 2
''') # executar o comando SQL
print(cursor.fetchall())

cursor.close()
conexao.close() # finalizar a conexão

[(2, 'Leonie', 'Köhler', None, 'Theodor-Heuss-Straße 34', 'Berlin', None, 'Germany', '70174', '+49 0711 2842222', None, 'leonekohler@surfeu.de', 5)]


## Passo 6 — Removendo registros (Delete)

O comando SQL `DELETE` remove linhas inteiras de uma tabela. Abaixo, um álbum é removido do `chinook.db` pelo seu `AlbumId`.

**⚠️ Cuidado com o `DELETE` sem `WHERE`** — sem essa cláusula, o comando apaga a tabela inteira, não só uma linha.

In [8]:
import pyodbc

dados_conexao = ("Driver={SQLite3 ODBC Driver};Server=localhost;Database=dataset/chinook.db")

conexao = pyodbc.connect(dados_conexao)

cursor = conexao.cursor()

In [9]:
cursor.execute('''
DELETE FROM albums WHERE AlbumId=2
''')

conexao.commit()

cursor.execute('''
SELECT * FROM albums
''') # executar o comando SQL
print(cursor.fetchall())

cursor.close()
conexao.close()

[(1, 'For Those About To Rock We Salute You', 1), (3, 'Restless and Wild', 2), (4, 'Let There Be Rock', 1), (5, 'Big Ones', 3), (6, 'Jagged Little Pill', 4), (7, 'Facelift', 5), (8, 'Warner 25 Anos', 6), (9, 'Plays Metallica By Four Cellos', 7), (10, 'Audioslave', 8), (11, 'Out Of Exile', 8), (12, 'BackBeat Soundtrack', 9), (13, 'The Best Of Billy Cobham', 10), (14, 'Alcohol Fueled Brewtality Live! [Disc 1]', 11), (15, 'Alcohol Fueled Brewtality Live! [Disc 2]', 11), (16, 'Black Sabbath', 12), (17, 'Black Sabbath Vol. 4 (Remaster)', 12), (18, 'Body Count', 13), (19, 'Chemical Wedding', 14), (20, 'The Best Of Buddy Guy - The Millenium Collection', 15), (21, 'Prenda Minha', 16), (22, 'Sozinho Remix Ao Vivo', 16), (23, 'Minha Historia', 17), (24, 'Afrociberdelia', 18), (25, 'Da Lama Ao Caos', 18), (26, 'Acústico MTV [Live]', 19), (27, 'Cidade Negra - Hits', 19), (28, 'Na Pista', 20), (29, 'Axé Bahia 2001', 21), (30, 'BBC Sessions [Disc 1] [Live]', 22), (31, 'Bongo Fury', 23), (32, 'Carnav

### ✏️ Atividade Prática 4 — Sua vez de programar (Delete)

**Contextualização:** a parceria de remasterização do Passo 3 (Atividade 1) foi cancelada — a Chinook Records pediu para remover o álbum `'Back in Black (Remaster)'` que você cadastrou, já que a versão final saiu diferente da testada.

**Comando:** conecte no `chinook.db` e remova, da tabela `albums`, o registro com `Title = 'Back in Black (Remaster)'`. Depois, confirme com um `SELECT` que ele não aparece mais.

In [10]:
# Conexão
import pyodbc

dados_conexao = ("Driver={SQLite3 ODBC Driver};Server=localhost;Database=dataset/chinook.db")

conexao = pyodbc.connect(dados_conexao)

cursor = conexao.cursor()

# Remoção
cursor.execute('''
DELETE FROM albums WHERE Title= 'Back in Black (Remaster)'
''')

conexao.commit()

# Consulta e fechamento do cursor/conexão
cursor.execute('''
SELECT * FROM albums
''') # executar o comando SQL
print(cursor.fetchall())

cursor.close()
conexao.close()

[(1, 'For Those About To Rock We Salute You', 1), (3, 'Restless and Wild', 2), (4, 'Let There Be Rock', 1), (5, 'Big Ones', 3), (6, 'Jagged Little Pill', 4), (7, 'Facelift', 5), (8, 'Warner 25 Anos', 6), (9, 'Plays Metallica By Four Cellos', 7), (10, 'Audioslave', 8), (11, 'Out Of Exile', 8), (12, 'BackBeat Soundtrack', 9), (13, 'The Best Of Billy Cobham', 10), (14, 'Alcohol Fueled Brewtality Live! [Disc 1]', 11), (15, 'Alcohol Fueled Brewtality Live! [Disc 2]', 11), (16, 'Black Sabbath', 12), (17, 'Black Sabbath Vol. 4 (Remaster)', 12), (18, 'Body Count', 13), (19, 'Chemical Wedding', 14), (20, 'The Best Of Buddy Guy - The Millenium Collection', 15), (21, 'Prenda Minha', 16), (22, 'Sozinho Remix Ao Vivo', 16), (23, 'Minha Historia', 17), (24, 'Afrociberdelia', 18), (25, 'Da Lama Ao Caos', 18), (26, 'Acústico MTV [Live]', 19), (27, 'Cidade Negra - Hits', 19), (28, 'Na Pista', 20), (29, 'Axé Bahia 2001', 21), (30, 'BBC Sessions [Disc 1] [Live]', 22), (31, 'Bongo Fury', 23), (32, 'Carnav

### ✅ O que você fez até aqui

- Instalou a biblioteca `pyodbc` e o driver ODBC do SQLite no seu computador (Passo 0).
- Entendeu quando faz sentido integrar Python e SQL (Passo 1).
- Conectou o Python a um banco SQLite de ponta a ponta e leu dados de 2 formas — manual e com `pd.read_sql` (Passos 2 e 4).
- Fez o CRUD completo por código: `INSERT` (Passo 3), `UPDATE` (Passo 5) e `DELETE` (Passo 6), praticando cada operação sozinho nas Atividades 1 a 4.